# Homework 3
## Nonlinear Programming
### Shaun Ketner

#### Modern statistics is mostly just convex optimization.

### 1. (Regression) Let n “ 10000 and p “ 100. Generate an example design matrix x P Rnˆp and a response variable y P R via the following.

In [1]:
import numpy as np
import pandas as pd
import cvxpy as cvx
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
sns.set()

In [2]:
n = 10000
p = 100
np.random.seed(0) #set seed
beta = np.random.normal(size=p) #the true coefficient vector
X = np.random.normal(size=(n, p))
X[:,0] = 1. #first column is 1s, so we have an intercept
y = X @ beta + .5*np.random.normal(size=n)

### For the following estimators, identify whether the optimization problem defining the estimator is convex. If it is, compute the estimator in CVXPY.

#### (a) The least squares estimator

Yes the least squares estimators is convex, the squared L2 norm is a classic convex quadratic function.

In [3]:
b=cvx.Variable(p)
obj =cvx.Minimize(cvx.sum_squares(X @ b - y))
prob=cvx.Problem(obj)
prob.solve()

Set parameter Username
Set parameter LicenseID to value 2786311
Academic license - for non-commercial use only - expires 2027-03-02


2470.5107088562117

#### (b) The least absolute deviation (lad) estimator

Yes the least absolute deviation estimator is convex, the L1 norm is a convex function.

In [4]:
b=cvx.Variable(p)
obj =cvx.Minimize(cvx.norm(X @ b - y, 1))
prob=cvx.Problem(obj)
prob.solve()

np.float64(3956.153021596928)

#### (c) The ridge regression estimator with penalty parameter (λ=1).

Yes the ridge regression estimator is convex because the sum of two convex function is strictly convex.

In [5]:
b=cvx.Variable(p)
lam= 1.0
obj =cvx.Minimize(cvx.sum_squares(X @ b - y) + lam * cvx.sum_squares(b))
prob=cvx.Problem(obj)
prob.solve()

np.float64(2572.5159062983585)

#### (d) The lasso estimator with penalty parameter γ=1.

Yes the LASSO estimator is convex, the sum of a convex squared L2 norm and convex L1 norm norm penalty.

In [6]:
b=cvx.Variable(p)
gam = 1.0
obj =cvx.Minimize(cvx.sum_squares(X @ b - y) + gam * cvx.norm(b, 1))
prob=cvx.Problem(obj)
prob.solve()

np.float64(2551.5410875131242)

#### (e) The elastic net estimator (with both penalty parameters =1).

The eleastic net estimator is convex, it is a linear combination of three convex norms/functions with positive coefficients, preserving convexity.

In [7]:
b=cvx.Variable(p)
obj =cvx.Minimize(cvx.sum_squares(X @ b - y) + cvx.sum_squares(b) + cvx.norm(b, 1))
prob=cvx.Problem(obj)
prob.solve()

np.float64(2653.5383168729577)

#### (f) The least squares estimator, but with the constraint that ˆβ vector is a unit vector.

The least squares estimator with a unit vector constraint is not convex because the feasible set defined by the constraint sum_squares(beta)=1 is the surface of a hypersphere. The surface of a sphere is a hollow shell, which is not a convex set. Although the objective function is convex.

#### (g) The ridge regression estimator, but with the constraint that (ˆβ >=0).

The ridge regression estimator witht the constraint beta>=0 is convex, because the objective is convex, and the inequality constraint beta>=0 restricts the solution to the non-negative orthant, which is a convex set.

In [8]:
beta_1 = cvx.Variable(p, nonneg=True)
lambda_ = 1.0
obj =cvx.Minimize(cvx.sum_squares(X @ beta_1 - y) + lambda_ * cvx.sum_squares(beta_1))
prob=cvx.Problem(obj)
prob.solve()

np.float64(468674.527240019)

### 2. (Binary Classification/Logistic Regression) Redefine your response variable in the above problem as follows.

In [9]:
np.random.seed(0) #reset the seed
y = np.random.binomial(n=1, p=scipy.special.expit(X @ beta))

##### Verify that this optimization problem is the maximum log-likehood estimator for the statistical model specified in the Python code.


#### Is this problem convex?

The term log(1+e^(xi*b)) is the softplus function, -log(1+e^(xi*b)) is concave and yi*xi*b is linear (convex and concave), and the sum of concave functions is concave. Maximizing a concave function is a convex problem, so the logistic regression objective is convex.

In [10]:
b=cvx.Variable(p)
log_likelihood = y@(X@b)-cvx.sum(cvx.logistic(X@b))
prob=cvx.Problem(cvx.Maximize(log_likelihood))
prob.solve()

np.float64(-1179.3474622718131)

### 3. (Multi-label Classification/Multinomial Regression) In this problem we will classify an observation into one of L different classes. Redefine β as a Rpˆl matrix as follows.

In [11]:
l = 20 #number of different classes
np.random.seed(0) #reset the seed
#the true coefficient matrix (one column for each class)
beta = np.random.normal(size=(p, l))
probs = scipy.special.softmax(X @ beta, axis=1)
y = np.array([np.random.choice(np.arange(l), p=probs[i]) for i in range(n)])

In [12]:
b=cvx.Variable((p, l))
Y = np.zeros((n, l))
Y[np.array(range(n)), y] = 1
log_likelihood = cvx.sum(cvx.multiply(Y, X@b)) - cvx.sum(cvx.log_sum_exp(X@b, axis=1))
prob=cvx.Problem(cvx.Maximize(log_likelihood))
prob.solve(solver=cvx.SCS, verbose=True, max_iters=2500)

(CVXPY) Jul 28 09:57:19 PM: Your problem has 2000 variables, 0 constraints, and 0 parameters.
(CVXPY) Jul 28 09:57:19 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 28 09:57:19 PM: DCP verification time: 0.0002 seconds.
(CVXPY) Jul 28 09:57:19 PM: Expression tree has 6 nodes.


(CVXPY) Jul 28 09:57:19 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 28 09:57:19 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 28 09:57:19 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 28 09:57:19 PM: Compiling problem (target solver=SCS).
(CVXPY) Jul 28 09:57:19 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> SCS
(CVXPY) Jul 28 09:57:19 PM: Applying reduction FlipObjective
(CVXPY) Jul 28 09:57:19 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 28 09:57:19 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 28 09:57:19 PM: Applying reduction EliminateZeroSized
(CVXPY) Jul 28 09:57:19 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 28 09:57:29 PM: Applying reduction SCS
(CVXPY) Jul 28 09:57:35 PM: Finished problem compilation (took 1.611e+01 seconds).
(CVXPY) Jul 28 09:57:35 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 212000, constraints m: 610000
cones: 	  l: linear vars: 10000
	  e: exp vars: 600000, dual exp vars: 0
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 2500, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 20600000, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  

(CVXPY) Jul 28 10:02:20 PM: Problem status: optimal
(CVXPY) Jul 28 10:02:20 PM: Optimal value: -2.415e+03
(CVXPY) Jul 28 10:02:20 PM: Compilation took 1.611e+01 seconds
(CVXPY) Jul 28 10:02:20 PM: Solver (including time spent in interface) took 2.851e+02 seconds


  1250| 1.65e-04  6.99e-03  1.93e-02  2.42e+03  1.00e-01  2.85e+02 
WARNING - large complementary slackness residual: 0.420442
------------------------------------------------------------------
status:  solved
timings: total: 2.85e+02s = setup: 4.61e+01s + solve: 2.39e+02s
	 lin-sys: 1.83e+02s, cones: 4.33e+01s, accel: 1.72e+00s
------------------------------------------------------------------
objective = 2415.645440
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(-2415.3255523203934)

In [13]:
#Dr. Bassett's way
obj = cvx.Maximize(cvx.sum(cvx.sum(cvx.multiply(X.T, b[:,y]), axis=0) - cvx.log_sum_exp(X@b, axis=1)))
prob = cvx.Problem(obj)
prob.solve(verbose=True, solver='SCS')

(CVXPY) Jul 28 10:02:20 PM: Your problem has 2000 variables, 0 constraints, and 0 parameters.
(CVXPY) Jul 28 10:02:20 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 28 10:02:20 PM: DCP verification time: 0.0002 seconds.
(CVXPY) Jul 28 10:02:20 PM: Expression tree has 7 nodes.
(CVXPY) Jul 28 10:02:20 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 28 10:02:20 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 28 10:02:20 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 28 10:02:20 PM: Compiling problem (target solver=SCS).
(CVXPY) Jul 28 10:02:20 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> SCS
(CVXPY) Jul 28 10:02:20 PM: Applying reduction FlipObjective
(CVXPY) Jul 28 10:02:20 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 28 10:02:2

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 28 10:02:30 PM: Applying reduction SCS
(CVXPY) Jul 28 10:02:36 PM: Finished problem compilation (took 1.611e+01 seconds).
(CVXPY) Jul 28 10:02:36 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 212000, constraints m: 610000
cones: 	  l: linear vars: 10000
	  e: exp vars: 600000, dual exp vars: 0
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 20600000, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |

(CVXPY) Jul 28 10:06:26 PM: Problem status: optimal
(CVXPY) Jul 28 10:06:26 PM: Optimal value: -2.415e+03
(CVXPY) Jul 28 10:06:26 PM: Compilation took 1.611e+01 seconds
(CVXPY) Jul 28 10:06:26 PM: Solver (including time spent in interface) took 2.300e+02 seconds


  1250| 1.65e-04  6.99e-03  1.93e-02  2.42e+03  1.00e-01  2.30e+02 
WARNING - large complementary slackness residual: 0.420442
------------------------------------------------------------------
status:  solved
timings: total: 2.30e+02s = setup: 4.68e+01s + solve: 1.83e+02s
	 lin-sys: 1.41e+02s, cones: 3.26e+01s, accel: 1.15e+00s
------------------------------------------------------------------
objective = 2415.645440
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(-2415.325552320396)